In [0]:

CREATE TABLE IF NOT EXISTS bootcamp_gabriel.silver.properties_silver 
USING DELTA
COMMENT 'Tabla de propiedades limpia y normalizada'
TBLPROPERTIES ('quality' = 'silver');


INSERT INTO bootcamp_gabriel.silver.properties_silver
WITH 
limpieza_base AS (
    SELECT 
        id,
        CAST(precio AS DECIMAL(15,2)) as precio,
        UPPER(moneda) as moneda, -- Normalizamos a mayúsculas
        CAST(ambientes AS INT) as ambientes,
        CAST(metros_cuadrados_totales AS DECIMAL(10,2)) as metros_totales,
        CAST(metros_cuadrados_cubiertos AS DECIMAL(10,2)) as metros_cubiertos,
        cochera,
        CAST(antiguedad AS INT) as antiguedad,
        zona as zona_original, -- Guardamos la original por si acaso
        -- Aquí podrías pegar el CASE WHEN de subzonas más adelante
        zona as partido, 
        ubicacion as region,
        url
    FROM bootcamp_gabriel.bronze.properties_bronze
    WHERE precio IS NOT NULL 
      AND precio > 0 
      AND moneda IN ('USD', 'ARS') -- Filtramos solo las monedas que nos sirven
      AND ambientes > 0
),

-- CTE 2: Agregamos Metadata (Punto 6)
con_metadata AS (
    SELECT 
        *,
        current_timestamp() as _processing_timestamp,
        'properties_bronze' as _source_table
    FROM limpieza_base
)

-- Inserción final
SELECT * FROM con_metadata;

In [0]:
-- 1. Borramos la tabla que quedó con la estructura "vacía" o mal armada
DROP TABLE IF EXISTS bootcamp_gabriel.silver.properties_silver;

-- 2. La creamos DE CERO usando el resultado de tu consulta. 
-- Esto hace que Databricks aprenda los tipos de datos (Decimal, Int, etc.) automáticamente.
CREATE TABLE bootcamp_gabriel.silver.properties_silver
USING DELTA
AS
WITH 
limpieza_base AS (
    SELECT 
        CAST(id AS DOUBLE) as id, -- Según tu error, el ID viene como Double
        CAST(precio AS DECIMAL(15,2)) as precio,
        UPPER(moneda) as moneda,
        CAST(CAST(ambientes AS DECIMAL) AS INT) as ambientes,
        CAST(metros_cuadrados_totales AS DECIMAL(10,2)) as metros_totales,
        CAST(metros_cuadrados_cubiertos AS DECIMAL(10,2)) as metros_cubiertos,
        cochera,
        CAST(CAST(antiguedad AS DECIMAL) AS INT) as antiguedad,
        zona as zona_original,
        zona as partido, 
        ubicacion as region,
        url
    FROM bootcamp_gabriel.bronze.properties_bronze
    WHERE precio IS NOT NULL 
      AND precio > 0 
      AND moneda IN ('USD', 'ARS')
      AND ambientes > 0
),
con_metadata AS (
    SELECT 
        *,
        current_timestamp() as _processing_timestamp,
        'properties_bronze' as _source_table
    FROM limpieza_base
)
SELECT * FROM con_metadata;

In [0]:
-- 1. Borramos la tabla para limpiar intentos fallidos
DROP TABLE IF EXISTS bootcamp_gabriel.silver.properties_silver;

-- 2. Creamos la tabla Silver (Punto 4, 5 y 6)
CREATE TABLE bootcamp_gabriel.silver.properties_silver
USING DELTA
AS
WITH limpieza_base AS (
    SELECT 
        id,
        CAST(precio AS DECIMAL(15,2)) as precio,
        UPPER(moneda) as moneda,
        -- DOBLE CAST: Primero a DECIMAL para que acepte el ".0" y luego a INT
        CAST(CAST(ambientes AS DECIMAL) AS INT) as ambientes,
        CAST(metros_cuadrados_totales AS DECIMAL(10,2)) as metros_totales,
        CAST(metros_cuadrados_cubiertos AS DECIMAL(10,2)) as metros_cubiertos,
        cochera,
        -- Lo mismo para antigüedad por las dudas
        CAST(CAST(antiguedad AS DECIMAL) AS INT) as antiguedad,
        zona as zona_original,
        zona as partido, 
        ubicacion as region,
        url
    FROM bootcamp_gabriel.bronze.properties_bronze
    WHERE precio IS NOT NULL 
      AND precio > 0 
      AND moneda IN ('USD', 'ARS')
      -- Filtramos ambientes usando la misma lógica de conversión
      AND CAST(CAST(ambientes AS DECIMAL) AS INT) > 0
),
con_metadata AS (
    SELECT 
        *,
        current_timestamp() as _processing_timestamp,
        'properties_bronze' as _source_table
    FROM limpieza_base
)
SELECT * FROM con_metadata;

In [0]:
-- 1. Borramos la tabla para limpiar cualquier rastro de error
DROP TABLE IF EXISTS bootcamp_gabriel.silver.properties_silver;

-- 2. Creamos la tabla Silver con la lógica que NO ROMPE
CREATE TABLE bootcamp_gabriel.silver.properties_silver
USING DELTA
AS
WITH limpieza_base AS (
    SELECT 
        id,
        -- Usamos TRY_CAST por si hay basura, así devuelve NULL en vez de error
        TRY_CAST(precio AS DECIMAL(15,2)) as precio,
        UPPER(moneda) as moneda,
        -- El truco final: Primero a DECIMAL para que se coma el ".0" y luego a INT
        CAST(TRY_CAST(ambientes AS DECIMAL) AS INT) as ambientes,
        TRY_CAST(metros_cuadrados_totales AS DECIMAL(10,2)) as metros_totales,
        TRY_CAST(metros_cuadrados_cubiertos AS DECIMAL(10,2)) as metros_cubiertos,
        cochera,
        CAST(TRY_CAST(antiguedad AS DECIMAL) AS INT) as antiguedad,
        zona as zona_original,
        zona as partido, 
        ubicacion as region,
        url
    FROM bootcamp_gabriel.bronze.properties_bronze
    WHERE precio IS NOT NULL 
      -- Usamos TRY_CAST también en el filtro para que no explote
      AND TRY_CAST(precio AS DECIMAL) > 0 
      AND moneda IN ('USD', 'ARS')
      AND CAST(TRY_CAST(ambientes AS DECIMAL) AS INT) > 0
),
con_metadata AS (
    SELECT 
        *,
        current_timestamp() as _processing_timestamp,
        'properties_bronze' as _source_table
    FROM limpieza_base
)
SELECT * FROM con_metadata;

In [0]:
SELECT count(*) FROM bootcamp_gabriel.bronze.properties_bronze

In [0]:
SELECT ambientes, TRY_CAST(ambientes AS DECIMAL) as test FROM bootcamp_gabriel.bronze.properties_bronze LIMIT 20

In [0]:
-- 1. Borramos la tabla para que no dé error de "ya existe" o "esquema viejo"
DROP TABLE IF EXISTS bootcamp_gabriel.silver.properties_silver;

-- 2. Creamos la tabla Silver de una
CREATE TABLE bootcamp_gabriel.silver.properties_silver
USING DELTA
AS
SELECT 
    id,
    TRY_CAST(precio AS DECIMAL(15,2)) as precio,
    UPPER(TRIM(moneda)) as moneda,
    -- El truco del doble CAST para que el "5.0" sea un 5
    CAST(TRY_CAST(ambientes AS DECIMAL) AS INT) as ambientes,
    TRY_CAST(metros_cuadrados_totales AS DECIMAL(10,2)) as metros_totales,
    TRY_CAST(metros_cuadrados_cubiertos AS DECIMAL(10,2)) as metros_cubiertos,
    cochera,
    CAST(TRY_CAST(antiguedad AS DECIMAL) AS INT) as antiguedad,
    zona as zona_original,
    zona as partido, 
    ubicacion as region,
    url,
    current_timestamp() as _processing_timestamp,
    'properties_bronze' as _source_table
FROM bootcamp_gabriel.bronze.properties_bronze
WHERE precio IS NOT NULL 
  AND TRY_CAST(precio AS DECIMAL) > 0;

In [0]:
-- Primero limpiamos para no duplicar si lo corrés dos veces
TRUNCATE TABLE bootcamp_gabriel.silver.properties_silver;

-- Insertamos la data con las transformaciones (CAST y TRY_CAST)
INSERT INTO bootcamp_gabriel.silver.properties_silver
SELECT 
    id,
    TRY_CAST(precio AS DECIMAL(15,2)),
    UPPER(TRIM(moneda)),
    CAST(TRY_CAST(ambientes AS DECIMAL) AS INT),
    TRY_CAST(metros_cuadrados_totales AS DECIMAL(10,2)),
    TRY_CAST(metros_cuadrados_cubiertos AS DECIMAL(10,2)),
    cochera,
    CAST(TRY_CAST(antiguedad AS DECIMAL) AS INT),
    zona, -- zona_original
    zona, -- partido (aquí iría el archivo de subzonas si lo tuvieras)
    ubicacion, -- region
    url,
    current_timestamp(),
    'properties_bronze'
FROM bootcamp_gabriel.bronze.properties_bronze
WHERE precio IS NOT NULL; -- Filtro básico para que entren filas

In [0]:
-- 1. Limpiamos la tabla para cargar de cero con las zonas nuevas
TRUNCATE TABLE bootcamp_gabriel.silver.properties_silver;

-- 2. Insertamos con toda la lógica de limpieza (Números + Zonas)
INSERT INTO bootcamp_gabriel.silver.properties_silver
SELECT 
    id,
    TRY_CAST(precio AS DECIMAL(15,2)) as precio,
    UPPER(TRIM(moneda)) as moneda,
    CAST(TRY_CAST(ambientes AS DECIMAL) AS INT) as ambientes,
    TRY_CAST(metros_cuadrados_totales AS DECIMAL(10,2)) as metros_totales,
    TRY_CAST(metros_cuadrados_cubiertos AS DECIMAL(10,2)) as metros_cubiertos,
    cochera,
    CAST(TRY_CAST(antiguedad AS DECIMAL) AS INT) as antiguedad,
    zona as zona_original,
    -- LÓGICA DE PARTIDO (El CASE WHEN que te pasaron)
    CASE
        WHEN zona = 'capital-federal' THEN 'capital federal'
        WHEN zona = 'bsas-gba-norte/vicente-lopez' THEN 'vicente lopez'
        WHEN zona = 'bsas-gba-norte/pilar' THEN 'pilar'
        WHEN zona = 'bsas-gba-norte/tigre' THEN 'tigre'
        WHEN zona = 'bsas-gba-norte/san-isidro' THEN 'san isidro'
        WHEN zona = 'bsas-gba-norte/general-san-martin' THEN 'general san martin'
        WHEN zona = 'bsas-gba-norte/san-fernando' THEN 'san fernando'
        WHEN zona = 'bsas-gba-norte/san-miguel' THEN 'san miguel'
        WHEN zona = 'bsas-gba-norte/malvinas-argentinas' THEN 'malvinas argentinas'
        WHEN zona = 'bsas-gba-norte/escobar' THEN 'escobar'
        WHEN zona = 'bsas-gba-oeste/caseros' THEN 'tres de febrero'
        WHEN zona = 'bsas-gba-oeste/castelar' THEN 'moron'
        WHEN zona = 'bsas-gba-oeste/general-rodriguez' THEN 'general rodriguez'
        WHEN zona = 'bsas-gba-oeste/hurlingham' THEN 'hurlingham'
        WHEN zona = 'bsas-gba-oeste/ituzaingo' THEN 'ituzaingo'
        WHEN zona = 'bsas-gba-oeste/la-matanza' THEN 'la matanza'
        WHEN zona = 'bsas-gba-oeste/merlo' THEN 'merlo'
        WHEN zona = 'bsas-gba-oeste/moreno' THEN 'moreno'
        WHEN zona = 'bsas-gba-oeste/moron' THEN 'moron'
        WHEN zona = 'bsas-gba-oeste/tres-de-febrero' THEN 'tres de febrero'
        WHEN zona = 'bsas-gba-sur/almirante-brown' THEN 'almirante brown'
        WHEN zona = 'bsas-gba-sur/avellaneda' THEN 'avellaneda'
        WHEN zona = 'bsas-gba-sur/berazategui' THEN 'berazategui'
        WHEN zona = 'bsas-gba-sur/esteban-echeverria' THEN 'esteban echeverria'
        WHEN zona = 'bsas-gba-sur/ezeiza' THEN 'ezeiza'
        WHEN zona = 'bsas-gba-sur/florencio-varela' THEN 'florencio varela'
        WHEN zona = 'bsas-gba-sur/lanus' THEN 'lanus'
        WHEN zona = 'bsas-gba-sur/la-plata' THEN 'la plata'
        WHEN zona = 'bsas-gba-sur/lomas-de-zamora' THEN 'lomas de zamora'
        WHEN zona = 'bsas-gba-sur/quilmes' THEN 'quilmes'
        WHEN zona RLIKE 'vicente-lopez' THEN 'vicente lopez'
        WHEN zona RLIKE 'pilar' THEN 'pilar'
        WHEN zona RLIKE 'general-san-martin' THEN 'general san martin'
        WHEN zona RLIKE 'tigre' THEN 'tigre'
        WHEN zona RLIKE 'san-fernando' THEN 'san fernando'
        WHEN zona RLIKE 'san-miguel' THEN 'san miguel'
        WHEN zona RLIKE 'malvinas-argentinas' THEN 'malvinas argentinas'
        WHEN zona RLIKE 'escobar' THEN 'escobar'
        WHEN zona RLIKE 'caseros' THEN 'tres de febrero'
        WHEN zona RLIKE 'castelar' THEN 'moron'
        WHEN zona RLIKE 'general-rodriguez' THEN 'general rodriguez'
        WHEN zona RLIKE 'hurlingham' THEN 'hurlingham'
        WHEN zona RLIKE 'ituzaingo' THEN 'ituzaingo'
        WHEN zona RLIKE 'la-matanza' THEN 'la matanza'
        WHEN zona RLIKE 'merlo' THEN 'merlo'
        WHEN zona RLIKE 'moreno' THEN 'moreno'
        WHEN zona RLIKE 'moron' THEN 'moron'
        WHEN zona RLIKE 'tres-de-febrero' THEN 'tres de febrero'
        WHEN zona RLIKE 'almirante-brown' THEN 'almirante brown'
        WHEN zona RLIKE 'avellaneda' THEN 'avellaneda'
        WHEN zona RLIKE 'berazategui' THEN 'berazategui'
        WHEN zona RLIKE 'berisso' THEN 'berisso'
        WHEN zona RLIKE 'ensenada' THEN 'ensenada'
        WHEN zona RLIKE 'esteban-echeverria' THEN 'esteban echeverria'
        WHEN zona RLIKE 'ezeiza' THEN 'ezeiza'
        WHEN zona RLIKE 'florencio-varela' THEN 'florencio varela'
        WHEN zona RLIKE 'lanus' THEN 'lanus'
        WHEN zona RLIKE 'la-plata' THEN 'la plata'
        WHEN zona RLIKE 'lomas-de-zamora' THEN 'lomas de zamora'
        WHEN zona RLIKE 'presidente-peron' THEN 'presidente peron'
        WHEN zona RLIKE 'quilmes' THEN 'quilmes'
        WHEN zona RLIKE 'san-vicente' THEN 'san vicente'
        ELSE 'no especifica'
    END AS partido,
    -- LÓGICA DE REGION
    CASE
        WHEN zona IS NULL THEN 'no especifica'
        WHEN zona = 'capital-federal' THEN 'capital federal'
        WHEN zona LIKE 'bsas-gba-norte/%' THEN 'gba zona norte'
        WHEN zona IN ('vicente-lopez', 'pilar', 'tigre', 'san-isidro', 'general-san-martin',
                     'san-fernando', 'san-miguel', 'jose-c-paz', 'malvinas-argentinas', 'escobar') THEN 'gba zona norte'
        WHEN zona LIKE 'bsas-gba-oeste/%' THEN 'gba zona oeste'
        WHEN zona IN ('caseros', 'castelar', 'general-rodriguez', 'hurlingham', 'ituzaingo',
                     'la-matanza', 'merlo', 'moreno', 'moron', 'tres-de-febrero') THEN 'gba zona oeste'
        WHEN zona LIKE 'bsas-gba-sur/%' THEN 'gba zona sur'
        WHEN zona IN ('almirante-brown', 'avellaneda', 'berazategui', 'berisso', 'ensenada',
                     'esteban-echeverria', 'ezeiza', 'florencio-varela', 'lanus', 'la-plata',
                     'lomas-de-zamora', 'presidente-peron', 'quilmes', 'san-vicente') THEN 'gba zona sur'
        WHEN zona LIKE 'gba-zona-norte--%' THEN 'gba zona norte'
        WHEN zona LIKE 'gba-zona-oeste--%' THEN 'gba zona oeste'
        WHEN zona LIKE 'gba-zona-sur--%' THEN 'gba zona sur'
        ELSE zona
    END AS region,
    url,
    current_timestamp(),
    'properties_bronze'
FROM bootcamp_gabriel.bronze.properties_bronze
WHERE precio IS NOT NULL;

In [0]:
-- 1. Limpiamos la tabla para asegurar que la carga sea limpia (Punto 4 de la semana)
TRUNCATE TABLE bootcamp_gabriel.silver.properties_silver;

-- 2. Insertamos con toda la lógica de limpieza, tipos y eliminación de duplicados (Punto 1)
INSERT INTO bootcamp_gabriel.silver.properties_silver
WITH raw_data_transformed AS (
    SELECT 
        id,
        TRY_CAST(precio AS DECIMAL(15,2)) as precio,
        UPPER(TRIM(moneda)) as moneda,
        CAST(TRY_CAST(ambientes AS DECIMAL) AS INT) as ambientes,
        TRY_CAST(metros_cuadrados_totales AS DECIMAL(10,2)) as metros_totales,
        TRY_CAST(metros_cuadrados_cubiertos AS DECIMAL(10,2)) as metros_cubiertos,
        cochera,
        CAST(TRY_CAST(antiguedad AS DECIMAL) AS INT) as antiguedad,
        zona as zona_original,
        -- LÓGICA DE PARTIDO
        CASE
            WHEN zona = 'capital-federal' THEN 'capital federal'
            WHEN zona = 'bsas-gba-norte/vicente-lopez' THEN 'vicente lopez'
            WHEN zona = 'bsas-gba-norte/pilar' THEN 'pilar'
            WHEN zona = 'bsas-gba-norte/tigre' THEN 'tigre'
            WHEN zona = 'bsas-gba-norte/san-isidro' THEN 'san isidro'
            WHEN zona = 'bsas-gba-norte/general-san-martin' THEN 'general san martin'
            WHEN zona = 'bsas-gba-norte/san-fernando' THEN 'san fernando'
            WHEN zona = 'bsas-gba-norte/san-miguel' THEN 'san miguel'
            WHEN zona = 'bsas-gba-norte/malvinas-argentinas' THEN 'malvinas argentinas'
            WHEN zona = 'bsas-gba-norte/escobar' THEN 'escobar'
            WHEN zona = 'bsas-gba-oeste/caseros' THEN 'tres de febrero'
            WHEN zona = 'bsas-gba-oeste/castelar' THEN 'moron'
            WHEN zona = 'bsas-gba-oeste/general-rodriguez' THEN 'general rodriguez'
            WHEN zona = 'bsas-gba-oeste/hurlingham' THEN 'hurlingham'
            WHEN zona = 'bsas-gba-oeste/ituzaingo' THEN 'ituzaingo'
            WHEN zona = 'bsas-gba-oeste/la-matanza' THEN 'la matanza'
            WHEN zona = 'bsas-gba-oeste/merlo' THEN 'merlo'
            WHEN zona = 'bsas-gba-oeste/moreno' THEN 'moreno'
            WHEN zona = 'bsas-gba-oeste/moron' THEN 'moron'
            WHEN zona = 'bsas-gba-oeste/tres-de-febrero' THEN 'tres de febrero'
            WHEN zona = 'bsas-gba-sur/almirante-brown' THEN 'almirante brown'
            WHEN zona = 'bsas-gba-sur/avellaneda' THEN 'avellaneda'
            WHEN zona = 'bsas-gba-sur/berazategui' THEN 'berazategui'
            WHEN zona = 'bsas-gba-sur/esteban-echeverria' THEN 'esteban echeverria'
            WHEN zona = 'bsas-gba-sur/ezeiza' THEN 'ezeiza'
            WHEN zona = 'bsas-gba-sur/florencio-varela' THEN 'florencio varela'
            WHEN zona = 'bsas-gba-sur/lanus' THEN 'lanus'
            WHEN zona = 'bsas-gba-sur/la-plata' THEN 'la plata'
            WHEN zona = 'bsas-gba-sur/lomas-de-zamora' THEN 'lomas de zamora'
            WHEN zona = 'bsas-gba-sur/quilmes' THEN 'quilmes'
            WHEN zona RLIKE 'vicente-lopez' THEN 'vicente lopez'
            WHEN zona RLIKE 'pilar' THEN 'pilar'
            WHEN zona RLIKE 'general-san-martin' THEN 'general san martin'
            WHEN zona RLIKE 'tigre' THEN 'tigre'
            WHEN zona RLIKE 'san-fernando' THEN 'san fernando'
            WHEN zona RLIKE 'san-miguel' THEN 'san miguel'
            WHEN zona RLIKE 'malvinas-argentinas' THEN 'malvinas argentinas'
            WHEN zona RLIKE 'escobar' THEN 'escobar'
            WHEN zona RLIKE 'caseros' THEN 'tres de febrero'
            WHEN zona RLIKE 'castelar' THEN 'moron'
            WHEN zona RLIKE 'general-rodriguez' THEN 'general rodriguez'
            WHEN zona RLIKE 'hurlingham' THEN 'hurlingham'
            WHEN zona RLIKE 'ituzaingo' THEN 'ituzaingo'
            WHEN zona RLIKE 'la-matanza' THEN 'la matanza'
            WHEN zona RLIKE 'merlo' THEN 'merlo'
            WHEN zona RLIKE 'moreno' THEN 'moreno'
            WHEN zona RLIKE 'moron' THEN 'moron'
            WHEN zona RLIKE 'tres-de-febrero' THEN 'tres de febrero'
            WHEN zona RLIKE 'almirante-brown' THEN 'almirante brown'
            WHEN zona RLIKE 'avellaneda' THEN 'avellaneda'
            WHEN zona RLIKE 'berazategui' THEN 'berazategui'
            WHEN zona RLIKE 'berisso' THEN 'berisso'
            WHEN zona RLIKE 'ensenada' THEN 'ensenada'
            WHEN zona RLIKE 'esteban-echeverria' THEN 'esteban echeverria'
            WHEN zona RLIKE 'ezeiza' THEN 'ezeiza'
            WHEN zona RLIKE 'florencio-varela' THEN 'florencio varela'
            WHEN zona RLIKE 'lanus' THEN 'lanus'
            WHEN zona RLIKE 'la-plata' THEN 'la plata'
            WHEN zona RLIKE 'lomas-de-zamora' THEN 'lomas de zamora'
            WHEN zona RLIKE 'presidente-peron' THEN 'presidente peron'
            WHEN zona RLIKE 'quilmes' THEN 'quilmes'
            WHEN zona RLIKE 'san-vicente' THEN 'san vicente'
            ELSE 'no especifica'
        END AS partido,
        -- LÓGICA DE REGION
        CASE
            WHEN zona IS NULL THEN 'no especifica'
            WHEN zona = 'capital-federal' THEN 'capital federal'
            WHEN zona LIKE 'bsas-gba-norte/%' THEN 'gba zona norte'
            WHEN zona IN ('vicente-lopez', 'pilar', 'tigre', 'san-isidro', 'general-san-martin',
                         'san-fernando', 'san-miguel', 'jose-c-paz', 'malvinas-argentinas', 'escobar') THEN 'gba zona norte'
            WHEN zona LIKE 'bsas-gba-oeste/%' THEN 'gba zona oeste'
            WHEN zona IN ('caseros', 'castelar', 'general-rodriguez', 'hurlingham', 'ituzaingo',
                         'la-matanza', 'merlo', 'moreno', 'moron', 'tres-de-febrero') THEN 'gba zona oeste'
            WHEN zona LIKE 'bsas-gba-sur/%' THEN 'gba zona sur'
            WHEN zona IN ('almirante-brown', 'avellaneda', 'berazategui', 'berisso', 'ensenada',
                         'esteban-echeverria', 'ezeiza', 'florencio-varela', 'lanus', 'la-plata',
                         'lomas-de-zamora', 'presidente-peron', 'quilmes', 'san-vicente') THEN 'gba zona sur'
            WHEN zona LIKE 'gba-zona-norte--%' THEN 'gba zona norte'
            WHEN zona LIKE 'gba-zona-oeste--%' THEN 'gba zona oeste'
            WHEN zona LIKE 'gba-zona-sur--%' THEN 'gba zona sur'
            ELSE zona
        END AS region,
        url,
        current_timestamp() as fecha_proceso,
        -- Función de ventana para identificar duplicados (Punto 1)
        ROW_NUMBER() OVER(PARTITION BY id ORDER BY current_timestamp() DESC) as rn
    FROM bootcamp_gabriel.bronze.properties_bronze
    WHERE precio IS NOT NULL
)
SELECT 
    id,
    precio,
    moneda,
    ambientes,
    metros_totales,
    metros_cubiertos,
    cochera,
    antiguedad,
    zona_original,
    partido,
    region,
    url,
    fecha_proceso,
    'properties_bronze' as fuente
FROM raw_data_transformed
WHERE rn = 1; -- Filtramos para que solo entre un registro por ID

In [0]:
TRUNCATE TABLE bootcamp_gabriel.silver.properties_silver;

-- 2. Insertamos con toda la lógica de limpieza, tipos y eliminación de duplicados
INSERT INTO bootcamp_gabriel.silver.properties_silver
WITH raw_data_transformed AS (
    SELECT 
        id,
        TRY_CAST(precio AS DECIMAL(15,2)) as precio,
        UPPER(TRIM(moneda)) as moneda,
        CAST(TRY_CAST(ambientes AS DECIMAL) AS INT) as ambientes,
        TRY_CAST(metros_cuadrados_totales AS DECIMAL(10,2)) as metros_totales,
        TRY_CAST(metros_cuadrados_cubiertos AS DECIMAL(10,2)) as metros_cubiertos,
        cochera,
        CAST(TRY_CAST(antiguedad AS DECIMAL) AS INT) as antiguedad,
        zona as zona_original,
        -- LÓGICA DE PARTIDO COMPLETA
        CASE
            WHEN zona = 'capital-federal' THEN 'capital federal'
            WHEN zona = 'bsas-gba-norte/vicente-lopez' THEN 'vicente lopez'
            WHEN zona = 'bsas-gba-norte/pilar' THEN 'pilar'
            WHEN zona = 'bsas-gba-norte/tigre' THEN 'tigre'
            WHEN zona = 'bsas-gba-norte/san-isidro' THEN 'san isidro'
            WHEN zona = 'bsas-gba-norte/general-san-martin' THEN 'general san martin'
            WHEN zona = 'bsas-gba-norte/san-fernando' THEN 'san fernando'
            WHEN zona = 'bsas-gba-norte/san-miguel' THEN 'san miguel'
            WHEN zona = 'bsas-gba-norte/malvinas-argentinas' THEN 'malvinas argentinas'
            WHEN zona = 'bsas-gba-norte/escobar' THEN 'escobar'
            WHEN zona = 'bsas-gba-oeste/caseros' THEN 'tres de febrero'
            WHEN zona = 'bsas-gba-oeste/castelar' THEN 'moron'
            WHEN zona = 'bsas-gba-oeste/general-rodriguez' THEN 'general rodriguez'
            WHEN zona = 'bsas-gba-oeste/hurlingham' THEN 'hurlingham'
            WHEN zona = 'bsas-gba-oeste/ituzaingo' THEN 'ituzaingo'
            WHEN zona = 'bsas-gba-oeste/la-matanza' THEN 'la matanza'
            WHEN zona = 'bsas-gba-oeste/merlo' THEN 'merlo'
            WHEN zona = 'bsas-gba-oeste/moreno' THEN 'moreno'
            WHEN zona = 'bsas-gba-oeste/moron' THEN 'moron'
            WHEN zona = 'bsas-gba-oeste/tres-de-febrero' THEN 'tres de febrero'
            WHEN zona = 'bsas-gba-sur/almirante-brown' THEN 'almirante brown'
            WHEN zona = 'bsas-gba-sur/avellaneda' THEN 'avellaneda'
            WHEN zona = 'bsas-gba-sur/berazategui' THEN 'berazategui'
            WHEN zona = 'bsas-gba-sur/esteban-echeverria' THEN 'esteban echeverria'
            WHEN zona = 'bsas-gba-sur/ezeiza' THEN 'ezeiza'
            WHEN zona = 'bsas-gba-sur/florencio-varela' THEN 'florencio varela'
            WHEN zona = 'bsas-gba-sur/lanus' THEN 'lanus'
            WHEN zona = 'bsas-gba-sur/la-plata' THEN 'la plata'
            WHEN zona = 'bsas-gba-sur/lomas-de-zamora' THEN 'lomas de zamora'
            WHEN zona = 'bsas-gba-sur/quilmes' THEN 'quilmes'
            WHEN zona RLIKE 'vicente-lopez' THEN 'vicente lopez'
            WHEN zona RLIKE 'pilar' THEN 'pilar'
            WHEN zona RLIKE 'general-san-martin' THEN 'general san martin'
            WHEN zona RLIKE 'tigre' THEN 'tigre'
            WHEN zona RLIKE 'san-fernando' THEN 'san fernando'
            WHEN zona RLIKE 'san-miguel' THEN 'san miguel'
            WHEN zona RLIKE 'malvinas-argentinas' THEN 'malvinas argentinas'
            WHEN zona RLIKE 'escobar' THEN 'escobar'
            WHEN zona RLIKE 'caseros' THEN 'tres de febrero'
            WHEN zona RLIKE 'castelar' THEN 'moron'
            WHEN zona RLIKE 'general-rodriguez' THEN 'general rodriguez'
            WHEN zona RLIKE 'hurlingham' THEN 'hurlingham'
            WHEN zona RLIKE 'ituzaingo' THEN 'ituzaingo'
            WHEN zona RLIKE 'la-matanza' THEN 'la matanza'
            WHEN zona RLIKE 'merlo' THEN 'merlo'
            WHEN zona RLIKE 'moreno' THEN 'moreno'
            WHEN zona RLIKE 'moron' THEN 'moron'
            WHEN zona RLIKE 'tres-de-febrero' THEN 'tres de febrero'
            WHEN zona RLIKE 'almirante-brown' THEN 'almirante brown'
            WHEN zona RLIKE 'avellaneda' THEN 'avellaneda'
            WHEN zona RLIKE 'berazategui' THEN 'berazategui'
            WHEN zona RLIKE 'berisso' THEN 'berisso'
            WHEN zona RLIKE 'ensenada' THEN 'ensenada'
            WHEN zona RLIKE 'esteban-echeverria' THEN 'esteban echeverria'
            WHEN zona RLIKE 'ezeiza' THEN 'ezeiza'
            WHEN zona RLIKE 'florencio-varela' THEN 'florencio varela'
            WHEN zona RLIKE 'lanus' THEN 'lanus'
            WHEN zona RLIKE 'la-plata' THEN 'la plata'
            WHEN zona RLIKE 'lomas-de-zamora' THEN 'lomas de zamora'
            WHEN zona RLIKE 'presidente-peron' THEN 'presidente peron'
            WHEN zona RLIKE 'quilmes' THEN 'quilmes'
            WHEN zona RLIKE 'san-vicente' THEN 'san vicente'
            ELSE 'no especifica'
        END AS partido,
        -- LÓGICA DE REGION COMPLETA
        CASE
            WHEN zona IS NULL THEN 'no especifica'
            WHEN zona = 'capital-federal' THEN 'capital federal'
            WHEN zona LIKE 'bsas-gba-norte/%' THEN 'gba zona norte'
            WHEN zona IN ('vicente-lopez', 'pilar', 'tigre', 'san-isidro', 'general-san-martin',
                         'san-fernando', 'san-miguel', 'jose-c-paz', 'malvinas-argentinas', 'escobar') THEN 'gba zona norte'
            WHEN zona LIKE 'bsas-gba-oeste/%' THEN 'gba zona oeste'
            WHEN zona IN ('caseros', 'castelar', 'general-rodriguez', 'hurlingham', 'ituzaingo',
                         'la-matanza', 'merlo', 'moreno', 'moron', 'tres-de-febrero') THEN 'gba zona oeste'
            WHEN zona LIKE 'bsas-gba-sur/%' THEN 'gba zona sur'
            WHEN zona IN ('almirante-brown', 'avellaneda', 'berazategui', 'berisso', 'ensenada',
                         'esteban-echeverria', 'ezeiza', 'florencio-varela', 'lanus', 'la-plata',
                         'lomas-de-zamora', 'presidente-peron', 'quilmes', 'san-vicente') THEN 'gba zona sur'
            WHEN zona LIKE 'gba-zona-norte--%' THEN 'gba zona norte'
            WHEN zona LIKE 'gba-zona-oeste--%' THEN 'gba zona oeste'
            WHEN zona LIKE 'gba-zona-sur--%' THEN 'gba zona sur'
            ELSE zona
        END AS region,
        url,
        current_timestamp() as fecha_proceso,
        ROW_NUMBER() OVER(PARTITION BY id ORDER BY current_timestamp() DESC) as rn
    FROM bootcamp_gabriel.bronze.properties_bronze
    WHERE precio IS NOT NULL
)
SELECT 
    id,
    precio,
    moneda,
    ambientes,
    metros_totales,
    metros_cubiertos,
    cochera,
    antiguedad,
    zona_original,
    partido,
    region,
    url,
    fecha_proceso,
    'properties_bronze' as fuente
FROM raw_data_transformed
WHERE rn = 1;

In [0]:
-- PUNTO 5: Carga Incremental usando MERGE INTO
MERGE INTO bootcamp_gabriel.silver.properties_silver AS target
USING (
    -- Usamos la misma lógica de limpieza que antes
    WITH cleaned_data AS (
        SELECT 
            id,
            TRY_CAST(precio AS DECIMAL(15,2)) as precio,
            UPPER(TRIM(moneda)) as moneda,
            CAST(TRY_CAST(ambientes AS DECIMAL) AS INT) as ambientes,
            TRY_CAST(metros_cuadrados_totales AS DECIMAL(10,2)) as metros_totales,
            TRY_CAST(metros_cuadrados_cubiertos AS DECIMAL(10,2)) as metros_cubiertos,
            cochera,
            CAST(TRY_CAST(antiguedad AS DECIMAL) AS INT) as antiguedad,
            zona as zona_original,
            url,
            current_timestamp() as fecha_proceso,
            ROW_NUMBER() OVER(PARTITION BY id ORDER BY current_timestamp() DESC) as rn
        FROM bootcamp_gabriel.bronze.properties_bronze
        WHERE precio IS NOT NULL
    )
    SELECT * FROM cleaned_data WHERE rn = 1
) AS source
ON target.id = source.id  -- La condición de cruce

-- Si el ID ya existe, actualizamos los datos (UPDATE)
WHEN MATCHED THEN
  UPDATE SET 
    target.precio = source.precio,
    target.moneda = source.moneda,
    target.ambientes = source.ambientes,
    target.metros_totales = source.metros_totales,
    target.metros_cubiertos = source.metros_cubiertos,
    target.fecha_proceso = source.fecha_proceso

-- Si el ID no existe, lo creamos (INSERT)
WHEN NOT MATCHED THEN
  INSERT (id, precio, moneda, ambientes, metros_totales, metros_cubiertos, cochera, antiguedad, zona_original, url, fecha_proceso, fuente)
  VALUES (source.id, source.precio, source.moneda, source.ambientes, source.metros_totales, source.metros_cubiertos, source.cochera, source.antiguedad, source.zona_original, source.url, source.fecha_proceso, 'properties_bronze');

In [0]:
MERGE INTO bootcamp_gabriel.silver.properties_silver AS target
USING (
    -- Usamos la misma lógica de limpieza que antes
    WITH cleaned_data AS (
        SELECT 
            id,
            TRY_CAST(precio AS DECIMAL(15,2)) as precio,
            UPPER(TRIM(moneda)) as moneda,
            CAST(TRY_CAST(ambientes AS DECIMAL) AS INT) as ambientes,
            TRY_CAST(metros_cuadrados_totales AS DECIMAL(10,2)) as metros_totales,
            TRY_CAST(metros_cuadrados_cubiertos AS DECIMAL(10,2)) as metros_cubiertos,
            cochera,
            CAST(TRY_CAST(antiguedad AS DECIMAL) AS INT) as antiguedad,
            zona as zona_original,
            url,
            current_timestamp() as fecha_proceso,
            ROW_NUMBER() OVER(PARTITION BY id ORDER BY current_timestamp() DESC) as rn
        FROM bootcamp_gabriel.bronze.properties_bronze
        WHERE precio IS NOT NULL
    )
    SELECT * FROM cleaned_data WHERE rn = 1
) AS source
ON target.id = source.id  -- La condición de cruce

-- Si el ID ya existe, actualizamos los datos (UPDATE)
WHEN MATCHED THEN
  UPDATE SET 
    target.precio = source.precio,
    target.moneda = source.moneda,
    target.ambientes = source.ambientes,
    target.metros_totales = source.metros_totales,
    target.metros_cubiertos = source.metros_cubiertos,
    target.fecha_proceso = source.fecha_proceso

-- Si el ID no existe, lo creamos (INSERT)
WHEN NOT MATCHED THEN
  INSERT (id, precio, moneda, ambientes, metros_totales, metros_cubiertos, cochera, antiguedad, zona_original, url, fecha_proceso, fuente)
  VALUES (source.id, source.precio, source.moneda, source.ambientes, source.metros_totales, source.metros_cubiertos, source.cochera, source.antiguedad, source.zona_original, source.url, source.fecha_proceso, 'properties_bronze');